# DATA 266 — Homework 12, Part 2: Knowledge Distillation (Corrected Objective)

**Student seed:** 1598  |  **Dataset:** AG News  |  **Teacher:** supplied local fine-tuned BERT  |  **Student:** `distilbert-base-uncased`

This is a **fresh, separately saved Part 2 rerun**. It does **not** overwrite or revise the earlier `T²` experiments. Every section follows **Question → Answer / code → saved evidence**, with a concise interpretation at the end.

**Important:** This notebook is based on the known assignment requirements and previous audit, not on a direct copy of your original notebook (which was not supplied here). Before running, check the original teacher checkpoint path, the professor's exact loss convention, and the baseline training hyperparameters. We use the assignment's explicitly written **unscaled** objective, $\mathcal{L}=\alpha CE+(1-\alpha)KL$; changing this formula changes the experiment.

**Colab:** Runtime → Change runtime type → **GPU** (T4/L4 or better). Start at Question 1 and run cells in order. A complete run trains six *distinct* student configurations (four temperatures and two additional alpha values; the $\alpha=0.5$ run is reused), so budget sufficient GPU time. Checkpoint files are saved to Google Drive immediately after each completed run.

## Question 1 — How do we configure Colab, the student-specific seed, and a persistent output location?

**Answer:** Mount Google Drive, set the path to the **complete** `teacher_model/` folder, and write new outputs to a separate directory. Do not point the output folder at your earlier experiment directory.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import os, sys, json, time, random, shutil, hashlib, gc, warnings

SEED = 1598
SID4 = 1598
assert SEED == SID4
SLICE = SID4 % 1000
HP_ID = SID4 % 6
CLS_A = SID4 % 10
CLS_B = (CLS_A + 1 + ((SID4 // 10) % 9)) % 10
print({'SID4': SID4, 'SEED': SEED, 'SLICE': SLICE, 'HP_ID': HP_ID, 'CLS_A': CLS_A, 'CLS_B': CLS_B})

# CHANGE THIS ONLY if you placed the supplied teacher_model folder elsewhere in Drive.
TEACHER_DIR = Path('/content/drive/MyDrive/teacher_model')
# Deliberately distinct from all earlier experiments, to preserve their evidence.
ROOT = Path('/content/drive/MyDrive/DATA266_HW12_UNSCALED_SEED1598')
ROOT.mkdir(parents=True, exist_ok=True)
for sub in ['manifests','teacher_logits','runs','results','figures','logs']:
    (ROOT/sub).mkdir(parents=True, exist_ok=True)
print('Teacher directory:', TEACHER_DIR)
print('Corrected results directory:', ROOT)
print('Teacher files:', sorted(p.name for p in TEACHER_DIR.glob('*')) if TEACHER_DIR.exists() else 'NOT FOUND')
assert (TEACHER_DIR/'config.json').exists(), 'Upload the entire teacher_model folder to Google Drive and set TEACHER_DIR.'
assert any((TEACHER_DIR/x).exists() for x in ('model.safetensors','pytorch_model.bin','model.safetensors.index.json','pytorch_model.bin.index.json')), 'Teacher weights missing. Upload the full teacher_model folder.'

## Question 2 — Which Python libraries and GPU will be used?

**Answer:** Install the required packages, record their versions, and verify CUDA. Because future package releases can change APIs, this notebook uses compatible version ranges and saves the actual resolved versions. Reuse the original assignment's exact versions if the professor requires them.

In [ ]:
%pip -q install 'transformers>=4.44,<5' 'datasets>=3,<5' 'accelerate>=1,<2' 'scikit-learn>=1.3,<2' 'safetensors>=0.4' 'matplotlib>=3.7' 'pandas>=2'

In [ ]:
import torch, numpy as np, pandas as pd, matplotlib.pyplot as plt
import transformers, datasets, sklearn
from datasets import load_dataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup, set_seed
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm
from datetime import datetime, timezone

assert torch.cuda.is_available(), 'GPU required. In Colab: Runtime → Change runtime type → GPU.'
DEVICE=torch.device('cuda')
print('GPU:',torch.cuda.get_device_name(0))
print('CUDA:',torch.version.cuda,'PyTorch:',torch.__version__)
print('Transformers:',transformers.__version__,'Datasets:',datasets.__version__,'scikit-learn:',sklearn.__version__)

def reseed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    set_seed(seed)
reseed()

def atomic_json(obj,path):
    path=Path(path); path.parent.mkdir(parents=True,exist_ok=True)
    tmp=path.with_name(path.name+'.tmp')
    tmp.write_text(json.dumps(obj,indent=2,default=str),encoding='utf-8')
    os.replace(tmp,path)

def sha256_file(path):
    h=hashlib.sha256()
    with open(path,'rb') as f:
        for chunk in iter(lambda:f.read(8*1024*1024),b''):h.update(chunk)
    return h.hexdigest()

ENV={'seed':SEED,'gpu':torch.cuda.get_device_name(0),'pytorch':torch.__version__,
     'cuda':torch.version.cuda,'transformers':transformers.__version__,
     'datasets':datasets.__version__,'scikit_learn':sklearn.__version__,'numpy':np.__version__}
atomic_json(ENV,ROOT/'manifests/environment.json')

## Question 3 — What are the dataset, model, and training settings?

**Answer:** AG News has 120,000 training and 7,600 test examples across World, Sports, Business and Sci/Tech. Make a **stratified** 108,000/12,000 training/validation split using seed 1598. The 7,600 official test examples must remain untouched until the final configuration is selected.

In [ ]:
LABELS=['World','Sports','Business','Sci/Tech']
STUDENT_ID='distilbert/distilbert-base-uncased'
MAX_LENGTH=256
EPOCHS=3
BATCH_SIZE=16
EVAL_BATCH_SIZE=64
LEARNING_RATE=2e-5
WEIGHT_DECAY=0.01
WARMUP_RATIO=0.06
CLIP_NORM=1.0
TRAIN_SEED=1598

ag=load_dataset('ag_news')
all_train=ag['train']
heldout_test=ag['test']
assert len(all_train)==120000 and len(heldout_test)==7600
idx=np.arange(len(all_train))
train_idx,val_idx=train_test_split(idx,test_size=0.10,stratify=np.array(all_train['label']),random_state=SEED)
train_idx=np.array(train_idx,dtype=np.int64)
val_idx=np.array(val_idx,dtype=np.int64)
assert len(train_idx)==108000 and len(val_idx)==12000
assert not set(train_idx).intersection(set(val_idx))
assert set(train_idx).union(set(val_idx))==set(range(120000))
np.save(ROOT/'manifests/train_indices_seed1598.npy',train_idx)
np.save(ROOT/'manifests/validation_indices_seed1598.npy',val_idx)
atomic_json({'seed':SEED,'dataset':'ag_news','train_size':len(train_idx),'validation_size':len(val_idx),
             'test_size':len(heldout_test),'train_indices_sha256':sha256_file(ROOT/'manifests/train_indices_seed1598.npy'),
             'validation_indices_sha256':sha256_file(ROOT/'manifests/validation_indices_seed1598.npy'),
             'label_names':LABELS},ROOT/'manifests/data_split_manifest.json')
print('Train:',len(train_idx),'Validation:',len(val_idx),'Official test:',len(heldout_test))
print('Class counts (train):',pd.Series(np.array(all_train['label'])[train_idx]).value_counts().sort_index().to_dict())

## Question 4 — How do we load and check the provided teacher checkpoint and student tokenizer?

**Answer:** Load the teacher strictly from the supplied local folder; do not silently replace it with another Hugging Face model. Validate that both classifiers use four classes. Each model uses its own tokenizer for inference.

In [ ]:
reseed()
teacher_tok=AutoTokenizer.from_pretrained(str(TEACHER_DIR),use_fast=True,local_files_only=True)
teacher=AutoModelForSequenceClassification.from_pretrained(str(TEACHER_DIR),local_files_only=True).to(DEVICE)
teacher.eval()
assert teacher.config.num_labels==4, f'Expected four teacher classes, found {teacher.config.num_labels}'
student_tok=AutoTokenizer.from_pretrained(STUDENT_ID,use_fast=True)
print('Teacher model:',type(teacher).__name__,'classes:',teacher.config.num_labels)
print('Teacher config id2label:',teacher.config.id2label)
print('Student tokenizer:',STUDENT_ID)

# IMPORTANT: Confirm teacher class ordering corresponds to AG News's [World, Sports, Business, Sci/Tech].
# A generic LABEL_0,... config is acceptable only if checkpoint training used the AG News numeric order.
assert len(LABELS)==4

## Question 5 — How do we extract teacher logits for all 120,000 AG News training examples?

**Answer:** Run the frozen teacher on every original AG News training row **in original index order**. Save teacher logits as `(120000, 4)` and record a checksum. Chunked storage allows this stage to resume after a Colab disconnect. The validation rows are cached only to satisfy the assignment's all-training-rows requirement; **student training uses the 108,000 training indices only**.

In [ ]:
TEACHER_LOGITS_DIR=ROOT/'teacher_logits'
CHUNK_SIZE=2000
TEACHER_BATCH=64

def infer_teacher_logits(texts,batch_size=TEACHER_BATCH):
    chunks=[]
    teacher.eval()
    with torch.inference_mode():
        for start in range(0,len(texts),batch_size):
            batch=texts[start:start+batch_size]
            tokens=teacher_tok(batch,return_tensors='pt',truncation=True,padding=True,max_length=MAX_LENGTH)
            tokens={k:v.to(DEVICE) for k,v in tokens.items()}
            logits=teacher(**tokens).logits.float().cpu().numpy()
            chunks.append(logits)
    return np.concatenate(chunks,axis=0).astype(np.float32)

for start in tqdm(range(0,len(all_train),CHUNK_SIZE),desc='Teacher logits chunks'):
    end=min(start+CHUNK_SIZE,len(all_train))
    path=TEACHER_LOGITS_DIR/f'logits_{start:06d}_{end:06d}.npy'
    if path.exists():
        try:
            saved=np.load(path,mmap_mode='r')
            if saved.shape==(end-start,4) and np.isfinite(saved).all():
                continue
        except Exception:
            pass
    texts=all_train.select(range(start,end))['text']
    values=infer_teacher_logits(texts)
    assert values.shape==(end-start,4) and np.isfinite(values).all()
    tmp=path.with_name(path.stem+'.tmp.npy')
    np.save(tmp,values)
    os.replace(tmp,path)

full_path=TEACHER_LOGITS_DIR/'teacher_all_120000_logits.npy'
expected_size=(120000,4)
if not full_path.exists() or np.load(full_path,mmap_mode='r').shape!=expected_size:
    full=np.lib.format.open_memmap(full_path,mode='w+',dtype=np.float32,shape=expected_size)
    for start in range(0,len(all_train),CHUNK_SIZE):
        end=min(start+CHUNK_SIZE,len(all_train))
        full[start:end]=np.load(TEACHER_LOGITS_DIR/f'logits_{start:06d}_{end:06d}.npy')
    full.flush();del full
teacher_logits=np.load(full_path,mmap_mode='r')
assert teacher_logits.shape==(120000,4) and np.isfinite(teacher_logits).all()
atomic_json({'dataset':'ag_news','row_order':'original train split indices 0..119999',
             'shape':[120000,4],'dtype':'float32','teacher_source':str(TEACHER_DIR),
             'sha256':sha256_file(full_path),'seed':SEED,'max_length':MAX_LENGTH},
            TEACHER_LOGITS_DIR/'teacher_logits_metadata.json')
print('Teacher logits:',teacher_logits.shape,'SHA256:',sha256_file(full_path))
# After caching, release teacher GPU memory for student training.
teacher.cpu(); del teacher
gc.collect();torch.cuda.empty_cache()

## Question 6 — What loss function does this corrected notebook use?

**Answer:** Use precisely the assignment's written objective, with **no $T^2$ multiplier**:

$$\mathcal{L}=\alpha\,\mathrm{CE}(z_s,y)+(1-\alpha)\,D_{KL}(\mathrm{softmax}(z_t/T)\parallel\mathrm{softmax}(z_s/T)).$$

The library's `kl_div(log_student, teacher_probs, reduction="batchmean")` evaluates $D_{KL}(p_t\parallel p_s)$. Do not blend old $T^2$ metrics into the new report.

In [ ]:
import torch.nn.functional as F

def distillation_loss(student_logits,teacher_logits,labels,T,alpha):
    assert T>0 and 0 <= alpha <= 1
    ce=F.cross_entropy(student_logits,labels)
    teacher_probs=F.softmax(teacher_logits.float()/T,dim=-1)
    student_logprobs=F.log_softmax(student_logits.float()/T,dim=-1)
    kl=F.kl_div(student_logprobs,teacher_probs,reduction='batchmean')
    total=alpha*ce+(1-alpha)*kl  # IMPORTANT: NO T**2 MULTIPLIER
    return total,ce.detach(),kl.detach()

# Small correctness smoke test (no training): compare with manual KL for normalized distributions.
a=torch.tensor([[1.5,0.0,-0.5,0.2]],device=DEVICE,requires_grad=True)
b=torch.tensor([[2.0,-1.0,0.3,0.4]],device=DEVICE)
y=torch.tensor([0],device=DEVICE)
l,ce,kl=distillation_loss(a,b,y,T=5,alpha=.5)
pt=F.softmax(b/5,dim=-1); ps=F.softmax(a/5,dim=-1)
manual_kl=(pt*(pt.log()-ps.log())).sum()
assert torch.allclose(kl,manual_kl,atol=1e-6)
assert torch.allclose(l,.5*ce+.5*kl,atol=1e-6)
print('Corrected unscaled loss smoke test passed:',{'total':float(l),'ce':float(ce),'kl':float(kl)})

## Question 7 — How are inputs prepared, and how is teacher-logit alignment guaranteed?

**Answer:** Tokenize once with the student tokenizer. Keep each row's original AG News index so that shuffling the training loader never misaligns the corresponding teacher logits.

In [ ]:
# Tokenize the original training split and official test split once (fast batched tokenizer).
def tokenize_batch(batch):
    return student_tok(batch['text'],truncation=True,max_length=MAX_LENGTH)
encoded_train=all_train.map(tokenize_batch,batched=True,batch_size=2000,desc='Tokenizing AG News original train')
encoded_test=heldout_test.map(tokenize_batch,batched=True,batch_size=2000,desc='Tokenizing AG News official test')

class IndexedNewsDataset(Dataset):
    def __init__(self,source,indices):
        self.source=source;self.indices=np.array(indices,dtype=np.int64)
    def __len__(self):return len(self.indices)
    def __getitem__(self,i):
        original_index=int(self.indices[i]); row=self.source[original_index]
        return {'input_ids':row['input_ids'],'attention_mask':row['attention_mask'],
                'label':int(row['label']),'original_index':original_index}

class NewsCollator:
    def __init__(self,tok):self.tok=tok
    def __call__(self,examples):
        x=self.tok.pad([{'input_ids':e['input_ids'],'attention_mask':e['attention_mask']} for e in examples],padding=True,return_tensors='pt')
        x['labels']=torch.tensor([e['label'] for e in examples],dtype=torch.long)
        x['original_index']=torch.tensor([e['original_index'] for e in examples],dtype=torch.long)
        return x

collate=NewsCollator(student_tok)
train_ds=IndexedNewsDataset(encoded_train,train_idx)
val_ds=IndexedNewsDataset(encoded_train,val_idx)
test_ds=IndexedNewsDataset(encoded_test,np.arange(len(encoded_test)))
assert train_ds[0]['original_index'] in set(train_idx)
assert np.isfinite(teacher_logits[train_ds[0]['original_index']]).all()
print('Student dataset prepared:',len(train_ds),len(val_ds),len(test_ds))

## Question 8 — How do we train a fresh DistilBERT student and save evidence after every experiment?

**Answer:** For each configuration, reset seed 1598, load a **fresh** `distilbert-base-uncased` classifier, train three epochs using the cached teacher logits, track validation accuracy and macro-F1, and save the best checkpoint. Completed runs are reused only if their configuration and saved model files are valid. If a run is interrupted, rerun only that incomplete configuration.

In [ ]:
from copy import deepcopy

COMMON_CONFIG={'student_id':STUDENT_ID,'seed':SEED,'epochs':EPOCHS,'batch_size':BATCH_SIZE,
  'eval_batch_size':EVAL_BATCH_SIZE,'lr':LEARNING_RATE,'weight_decay':WEIGHT_DECAY,
  'warmup_ratio':WARMUP_RATIO,'max_length':MAX_LENGTH,'clip_norm':CLIP_NORM,
  'loss_formula':'alpha*CE+(1-alpha)*KL (NO T^2)','train_size':len(train_ds),'val_size':len(val_ds),
  'split_manifest_sha256':sha256_file(ROOT/'manifests/data_split_manifest.json'),
  'teacher_logits_sha256':sha256_file(full_path)}
atomic_json(COMMON_CONFIG,ROOT/'manifests/training_config.json')

def new_student():
    reseed(SEED)
    m=AutoModelForSequenceClassification.from_pretrained(STUDENT_ID,num_labels=4,
        id2label={i:x for i,x in enumerate(LABELS)},label2id={x:i for i,x in enumerate(LABELS)})
    return m.to(DEVICE)

@torch.inference_mode()
def evaluate_student(model,dataset,batch_size=EVAL_BATCH_SIZE,return_predictions=False):
    model.eval(); loader=DataLoader(dataset,batch_size=batch_size,shuffle=False,collate_fn=collate)
    ys=[];preds=[];indices=[]
    for batch in loader:
        labels=batch.pop('labels');ix=batch.pop('original_index')
        logits=model(**{k:v.to(DEVICE) for k,v in batch.items()}).logits
        ys.extend(labels.tolist());preds.extend(logits.argmax(-1).cpu().tolist());indices.extend(ix.tolist())
    acc=float(accuracy_score(ys,preds)); mf1=float(f1_score(ys,preds,average='macro'))
    return ({'accuracy':acc,'macro_f1':mf1,'sample_count':len(ys)},
            (np.array(ys),np.array(preds),np.array(indices)) if return_predictions else None)

def is_valid_saved_run(run_dir,cfg):
    status=run_dir/'run_summary.json'; ckpt=run_dir/'best_model'
    if not status.exists() or not (ckpt/'config.json').exists():return False
    if not any((ckpt/f).exists() for f in ('model.safetensors','pytorch_model.bin')):return False
    try:
        old=json.loads(status.read_text())
        return old.get('configuration')==cfg and old.get('status')=='complete'
    except Exception:return False

def train_one(T,alpha,run_name):
    run_dir=ROOT/'runs'/run_name;run_dir.mkdir(parents=True,exist_ok=True)
    cfg={**COMMON_CONFIG,'temperature':float(T),'alpha':float(alpha),'name':run_name}
    if is_valid_saved_run(run_dir,cfg):
        print('Using previously COMPLETED compatible run:',run_name)
        return json.loads((run_dir/'run_summary.json').read_text())
    if (run_dir/'run_summary.json').exists():
        raise RuntimeError(f'Existing mismatched run directory: {run_dir}. Do not overwrite. Resolve path manually.')
    atomic_json(cfg,run_dir/'run_config.json')
    model=new_student()
    train_loader=DataLoader(train_ds,batch_size=BATCH_SIZE,shuffle=True,collate_fn=collate,
        generator=torch.Generator().manual_seed(SEED))
    optimizer=torch.optim.AdamW(model.parameters(),lr=LEARNING_RATE,weight_decay=WEIGHT_DECAY)
    steps=len(train_loader)*EPOCHS
    warmup=int(steps*WARMUP_RATIO)
    scheduler=get_linear_schedule_with_warmup(optimizer,num_warmup_steps=warmup,num_training_steps=steps)
    history=[]; best_acc=-1.;best_epoch=None
    start_wall=time.time()
    for epoch in range(1,EPOCHS+1):
        model.train();tot=0.;totce=0.;totkl=0.;seen=0
        for batch in tqdm(train_loader,desc=f'{run_name} epoch {epoch}/{EPOCHS}',leave=False):
            labels=batch.pop('labels').to(DEVICE)
            original=batch.pop('original_index').numpy()
            tlog=torch.from_numpy(np.asarray(teacher_logits[original],dtype=np.float32).copy()).to(DEVICE)
            inputs={k:v.to(DEVICE) for k,v in batch.items()}
            optimizer.zero_grad(set_to_none=True)
            slog=model(**inputs).logits
            loss,ce,kl=distillation_loss(slog,tlog,labels,T,alpha)
            if not torch.isfinite(loss):raise RuntimeError(f'Nonfinite loss: {run_name} epoch {epoch}')
            loss.backward();torch.nn.utils.clip_grad_norm_(model.parameters(),CLIP_NORM)
            optimizer.step();scheduler.step()
            b=labels.size(0);seen+=b;tot+=float(loss.item())*b;totce+=float(ce)*b;totkl+=float(kl)*b
        val,_=evaluate_student(model,val_ds)
        row={'epoch':epoch,'train_total_loss':tot/seen,'train_ce_loss':totce/seen,
             'train_unscaled_kl':totkl/seen,'val_accuracy':val['accuracy'],'val_macro_f1':val['macro_f1']}
        history.append(row)
        atomic_json(history,run_dir/'training_history.json')
        pd.DataFrame(history).to_csv(run_dir/'training_history.csv',index=False)
        print(run_name,row,flush=True)
        if val['accuracy']>best_acc:
            best_acc=val['accuracy'];best_epoch=epoch
            ckpt=run_dir/'best_model';ckpt.mkdir(parents=True,exist_ok=True)
            model.save_pretrained(str(ckpt),safe_serialization=True)
            student_tok.save_pretrained(str(ckpt))
            atomic_json({'best_epoch':epoch,'best_validation_accuracy':best_acc,'T':T,'alpha':alpha},
                        run_dir/'best_checkpoint.json')
    summary={'status':'complete','configuration':cfg,'best_epoch':best_epoch,
             'best_validation_accuracy':best_acc,
             'best_validation_macro_f1':next(r['val_macro_f1'] for r in history if r['epoch']==best_epoch),
             'wall_time_seconds':time.time()-start_wall,'history':history,
             'best_model_path':str(run_dir/'best_model')}
    atomic_json(summary,run_dir/'run_summary.json')
    del model,optimizer,scheduler,train_loader
    gc.collect();torch.cuda.empty_cache()
    return summary

print('Training function ready. Fresh initialization is performed for each distinct run.')

## Question 9 — Temperature sweep: What happens for $T=1,2,5,10$ at $\alpha=0.5$?

**Answer:** Train four fresh students. Hold all other settings fixed. Select the best temperature using **validation accuracy only**; never use test accuracy to choose T.

In [ ]:
temperature_rows=[]
for T in [1,2,5,10]:
    summary=train_one(T=T,alpha=0.5,run_name=f'T{T}_alpha050')
    temperature_rows.append({'T':T,'alpha':0.5,'best_epoch':summary['best_epoch'],
         'validation_accuracy':summary['best_validation_accuracy'],
         'validation_macro_f1':summary['best_validation_macro_f1'],
         'run_name':f'T{T}_alpha050'})
temperature_df=pd.DataFrame(temperature_rows)
temperature_df.to_csv(ROOT/'results/temperature_sweep_comparison.csv',index=False)
# Deterministic tie-break: accuracy first, then macro-F1, then smaller temperature.
best_temp_row=temperature_df.sort_values(['validation_accuracy','validation_macro_f1','T'],ascending=[False,False,True]).iloc[0]
BEST_T=int(best_temp_row['T'])
atomic_json({'selected_temperature':BEST_T,'selection_metric':'validation_accuracy',
             'tie_break':'validation_macro_f1 then smaller T',
             'sweep_records':temperature_rows},ROOT/'results/temperature_selection.json')
print(temperature_df.to_string(index=False))
print('VALIDATION-SELECTED best T:',BEST_T)

## Question 10 — Alpha sweep: What happens for $\alpha=0.25,0.50,0.75$ at the selected temperature?

**Answer:** Use the validation-selected temperature from Question 9; do not assume the old run's T=5 is still optimal. Train fresh students at 0.25 and 0.75. The existing **fresh** T=BEST_T, alpha=0.50 run meets the third condition, so reuse its results rather than retraining the identical configuration.

In [ ]:
alpha_rows=[]
for alpha in [0.25,0.50,0.75]:
    run_name=f'T{BEST_T}_alpha{int(round(alpha*100)):03d}'
    summary=train_one(T=BEST_T,alpha=alpha,run_name=run_name)
    alpha_rows.append({'T':BEST_T,'alpha':alpha,'best_epoch':summary['best_epoch'],
         'validation_accuracy':summary['best_validation_accuracy'],
         'validation_macro_f1':summary['best_validation_macro_f1'],'run_name':run_name})
alpha_df=pd.DataFrame(alpha_rows)
alpha_df.to_csv(ROOT/'results/alpha_sweep_comparison.csv',index=False)
best_alpha_row=alpha_df.sort_values(['validation_accuracy','validation_macro_f1','alpha'],ascending=[False,False,True]).iloc[0]
BEST_ALPHA=float(best_alpha_row['alpha'])
BEST_RUN=str(best_alpha_row['run_name'])
SELECTED_PATH=ROOT/'runs'/BEST_RUN/'best_model'
assert (SELECTED_PATH/'config.json').exists()
atomic_json({'best_temperature':BEST_T,'best_alpha':BEST_ALPHA,'selected_run':BEST_RUN,
             'best_model_path':str(SELECTED_PATH),'selection_metric':'validation_accuracy',
             'all_alpha_records':alpha_rows},ROOT/'results/selected_final_student.json')
print(alpha_df.to_string(index=False))
print('FINAL VALIDATION-SELECTED CONFIGURATION:',BEST_RUN)
print('Selected checkpoint:',SELECTED_PATH)

## Question 11 — What do the training-loss and validation-accuracy curves show?

**Answer:** Plot genuine epoch-level histories from saved run data. This section creates two figures and an aggregate history table; no values are hand-entered.

In [ ]:
history_rows=[]
for p in sorted((ROOT/'runs').glob('*/training_history.json')):
    rows=json.loads(p.read_text())
    for row in rows:history_rows.append({'configuration':p.parent.name,**row})
history_df=pd.DataFrame(history_rows)
assert not history_df.empty
history_df.to_csv(ROOT/'results/all_training_histories.csv',index=False)
for metric,title,filename,ylabel in [('train_total_loss','Student training loss (unscaled KD objective)','training_loss_curves.png','Loss'),
                                      ('val_accuracy','Validation accuracy by epoch','validation_accuracy_curves.png','Accuracy')]:
    plt.figure(figsize=(9,5.5))
    for key,g in history_df.groupby('configuration'):
        plt.plot(g['epoch'],g[metric],marker='o',label=key)
    plt.title(title);plt.xlabel('Epoch');plt.ylabel(ylabel);plt.xticks([1,2,3]);plt.grid(alpha=.25)
    plt.legend(fontsize=8,ncol=2);plt.tight_layout()
    plt.savefig(ROOT/'figures'/filename,dpi=200,bbox_inches='tight')
    plt.show();plt.close()
print('History and plots saved:',ROOT/'results/all_training_histories.csv')

## Question 12 — How do we perform the final held-out AG News test evaluation?

**Answer:** Only **after** selecting T and alpha using validation, evaluate the original teacher and selected student on the official 7,600-example test partition. Save paired predictions, per-class accuracy, a confusion matrix and at least five genuine teacher-correct/student-wrong examples. No hyperparameters are selected from test performance.

In [ ]:
def make_teacher_dataset():
    te=heldout_test.map(lambda b:teacher_tok(b['text'],truncation=True,max_length=MAX_LENGTH),batched=True,batch_size=2000,desc='Tokenize AG News test for teacher')
    return IndexedNewsDataset(te,np.arange(len(te)))

teacher=AutoModelForSequenceClassification.from_pretrained(str(TEACHER_DIR),local_files_only=True).to(DEVICE).eval()
student=AutoModelForSequenceClassification.from_pretrained(str(SELECTED_PATH),local_files_only=True).to(DEVICE).eval()

@torch.inference_mode()
def evaluate_generic_model(model,data,tokenizer,batch_size=64):
    loader=DataLoader(data,batch_size=batch_size,shuffle=False,collate_fn=NewsCollator(tokenizer))
    labels=[]; predictions=[]; indices=[]
    model.eval()
    for batch in tqdm(loader,desc='Final test evaluation',leave=False):
        y=batch.pop('labels'); idx=batch.pop('original_index')
        logits=model(**{k:v.to(DEVICE) for k,v in batch.items()}).logits
        labels.extend(y.tolist());predictions.extend(logits.argmax(dim=-1).cpu().tolist());indices.extend(idx.tolist())
    return np.array(labels),np.array(predictions),np.array(indices)

teacher_test_ds=make_teacher_dataset()
yt,pt,it=evaluate_generic_model(teacher,teacher_test_ds,teacher_tok)
ys,ps,is_=evaluate_generic_model(student,test_ds,student_tok)
assert len(yt)==len(ys)==7600
assert np.array_equal(it,is_) and np.array_equal(yt,ys), 'Teacher/student test alignment mismatch'
labels=yt

def classification_summary(y,p):
    return {'accuracy':float(accuracy_score(y,p)),
            'macro_f1':float(f1_score(y,p,average='macro')),
            'test_examples':len(y),
            'per_class_accuracy':{name:float(np.mean(p[y==i]==i)) for i,name in enumerate(LABELS)}}

tmetrics=classification_summary(labels,pt)
smetrics=classification_summary(labels,ps)
paired=pd.DataFrame({'test_index':it,'text':list(heldout_test['text']),
    'true_label':labels,'teacher_prediction':pt,'student_prediction':ps})
paired['true_class']=paired['true_label'].map(dict(enumerate(LABELS)))
paired['teacher_class']=paired['teacher_prediction'].map(dict(enumerate(LABELS)))
paired['student_class']=paired['student_prediction'].map(dict(enumerate(LABELS)))
paired.to_csv(ROOT/'results/paired_test_predictions.csv',index=False)
failures=paired[(paired.teacher_prediction==paired.true_label)&(paired.student_prediction!=paired.true_label)].copy()
failures.to_csv(ROOT/'results/all_teacher_correct_student_wrong.csv',index=False)
failures.head(5).to_csv(ROOT/'results/five_failure_examples.csv',index=False)
class_rows=[]
for i,name in enumerate(LABELS):
    class_rows.append({'class':name,'support':int(np.sum(labels==i)),
        'teacher_accuracy':tmetrics['per_class_accuracy'][name],
        'student_accuracy':smetrics['per_class_accuracy'][name],
        'difference_student_minus_teacher':smetrics['per_class_accuracy'][name]-tmetrics['per_class_accuracy'][name]})
perclass=pd.DataFrame(class_rows)
perclass.to_csv(ROOT/'results/per_class_accuracy.csv',index=False)
conf=confusion_matrix(labels,ps,labels=[0,1,2,3]);pd.DataFrame(conf,index=LABELS,columns=LABELS).to_csv(ROOT/'results/student_confusion_matrix.csv')
atomic_json({'teacher':tmetrics,'student':smetrics,'selected_run':BEST_RUN,
             'teacher_correct_student_wrong_count':len(failures)},ROOT/'results/final_test_metrics.json')
print('Teacher test accuracy:',f"{100*tmetrics['accuracy']:.4f}%")
print('Selected student test accuracy:',f"{100*smetrics['accuracy']:.4f}%")
print('Teacher correct / student wrong:',len(failures))
print(perclass.to_string(index=False))
print('Five genuine examples:')
display(failures[['test_index','true_class','teacher_class','student_class','text']].head(5))
# Free GPU before measuring each model separately.
teacher.cpu();student.cpu();del teacher,student
gc.collect();torch.cuda.empty_cache()

## Question 13 — How do teacher and student latency, GPU memory, and disk size compare?

**Answer:** Benchmark both models under identical Tesla T4/L4 conditions, **one at a time**, using 20 warm-up passes and 100 synchronized, batch-1 forward passes. Report p50/p95 (ms), peak allocated GPU memory, and **both complete directory size and model-weight-only size**, with correct decimal MB versus binary MiB units. These measurements exclude preprocessing and model loading.

In [ ]:
BENCH_TEXTS=list(heldout_test['text'][:100]);assert len(BENCH_TEXTS)==100

def directory_byte_sizes(path):
    path=Path(path)
    fs=[p for p in path.rglob('*') if p.is_file()]
    weights=[p for p in fs if p.name.endswith(('.safetensors','.bin')) and not p.name.startswith('optimizer')]
    total=sum(p.stat().st_size for p in fs)
    weight_bytes=sum(p.stat().st_size for p in weights)
    return {'directory_size_bytes':total,'directory_size_MB':total/1e6,'directory_size_MiB':total/(1024**2),
            'weight_size_bytes':weight_bytes,'weight_size_MB':weight_bytes/1e6,
            'weight_size_MiB':weight_bytes/(1024**2),'file_count':len(fs)}

def benchmark_model(model_dir,tokenizer,texts=BENCH_TEXTS,warmups=20,repetitions=100):
    torch.cuda.empty_cache();gc.collect()
    model=AutoModelForSequenceClassification.from_pretrained(str(model_dir),local_files_only=True).to(DEVICE).eval()
    encoded=tokenizer(texts,return_tensors='pt',padding='max_length',truncation=True,max_length=MAX_LENGTH)
    torch.cuda.reset_peak_memory_stats()
    timings=[]
    with torch.inference_mode():
        for j in range(warmups+repetitions):
            i=j%len(texts)
            inputs={k:v[i:i+1].to(DEVICE) for k,v in encoded.items()}
            torch.cuda.synchronize()
            start=time.perf_counter()
            _=model(**inputs).logits
            torch.cuda.synchronize()
            delta=(time.perf_counter()-start)*1000
            if j>=warmups:timings.append(delta)
    peak_gib=torch.cuda.max_memory_allocated()/(1024**3)
    peak_gb=torch.cuda.max_memory_allocated()/1e9
    data={'p50_latency_ms':float(np.percentile(timings,50)),'p95_latency_ms':float(np.percentile(timings,95)),
          'peak_allocated_memory_GiB':float(peak_gib),'peak_allocated_memory_GB':float(peak_gb),
          'warmups':warmups,'repetitions':repetitions,'batch_size':1,'sequence_length':MAX_LENGTH,
          'gpu':torch.cuda.get_device_name(0),**directory_byte_sizes(model_dir)}
    del model,inputs,encoded
    gc.collect();torch.cuda.empty_cache()
    return data

teacher_bench=benchmark_model(TEACHER_DIR,teacher_tok)
student_bench=benchmark_model(SELECTED_PATH,student_tok)
benchmark={'teacher':teacher_bench,'student':student_bench,'hardware':ENV['gpu'],
           'measurement_scope':'synchronized model forward pass; excludes tokenization/loading',
           'matched_inputs':'first 100 AG News official test texts; tokenizer-specific encodings; max length 256'}
atomic_json(benchmark,ROOT/'results/teacher_student_benchmark.json')
bench_df=pd.DataFrame.from_dict({'Teacher':teacher_bench,'Selected student':student_bench},orient='index')
bench_df.to_csv(ROOT/'results/teacher_student_benchmark.csv')
display(bench_df[['p50_latency_ms','p95_latency_ms','peak_allocated_memory_GiB','directory_size_MB','weight_size_MB']])

## Question 14 — Which artifacts and metrics must be retained for the submission and report?

**Answer:** Save a compact summary and a human-readable run log derived from actual saved files. The full evidence is in `ROOT/runs`, `ROOT/results`, `ROOT/figures`, `ROOT/manifests`, and `ROOT/teacher_logits`. Never write the old experiment's numbers into this new report.

In [ ]:
summary={'objective':'alpha*CE+(1-alpha)*KL (unscaled; no T^2)',
 'seed':SEED,'best_temperature':BEST_T,'best_alpha':BEST_ALPHA,'selected_run':BEST_RUN,
 'temperature_sweep':temperature_rows,'alpha_sweep':alpha_rows,
 'teacher_test':tmetrics,'selected_student_test':smetrics,
 'benchmark':benchmark,'teacher_correct_student_wrong_count':len(failures),
 'paths':{'root':str(ROOT),'selected_checkpoint':str(SELECTED_PATH),
          'teacher_model':str(TEACHER_DIR),'paired_predictions':str(ROOT/'results/paired_test_predictions.csv')}}
atomic_json(summary,ROOT/'results/submission_summary.json')

log_lines=[f'DATA 266 HW12 Part 2 — unscaled KD objective | seed {SEED}',
  'This file is an automatically generated summary from saved notebook results, not a verbatim stdout transcript.',
  f'GPU: {ENV["gpu"]}',f'Teacher logits: {full_path}',
  'Temperature sweep (alpha=0.5):',temperature_df.to_string(index=False),
  f'Selected temperature: {BEST_T}', 'Alpha sweep:',alpha_df.to_string(index=False),
  f'Selected alpha: {BEST_ALPHA}',
  'Teacher test metrics: '+json.dumps(tmetrics),
  'Student test metrics: '+json.dumps(smetrics),
  'Benchmark: '+json.dumps(benchmark),
  'Per-class accuracy:',perclass.to_string(index=False),
  f'Teacher-correct/student-wrong count: {len(failures)}',
  'These observations were obtained with the unscaled KL loss; older T² results are not reused.']
(ROOT/'logs/RUN_LOG_DISTILLATION_UNSCALED_SUMMARY.txt').write_text('\n\n'.join(log_lines),encoding='utf-8')
print('Complete. Final results directory:',ROOT)
print('Selected final student:',SELECTED_PATH)
print('Generated files:')
for p in sorted((ROOT/'results').glob('*')): print(' -',p.name)

## Question 15 — What should we conclude from the corrected experiments?

**Answer / interpretation (complete only AFTER running all previous cells):**

1. Read `results/temperature_sweep_comparison.csv` and state which **validation** temperature won; do not carry over the old T=5 automatically.
2. Read `results/alpha_sweep_comparison.csv` and state which **validation** alpha won; do not carry over the old 0.25 automatically.
3. Read `results/final_test_metrics.json` and compare the teacher and final selected student on test accuracy and macro-F1.
4. Read `results/teacher_student_benchmark.csv` to compare p50/p95 latency, peak **allocated** GPU memory and comparable directory/weight size in precisely labeled units.
5. Use `results/per_class_accuracy.csv` to identify the class with the largest decrease. Cite five **real** examples in `results/five_failure_examples.csv` and explain observed topic overlaps without making unsupported causal claims.
6. Compare these new distillation results with the existing GPTQ experiment **only qualitatively** because the tasks/models/hardware differ. In the IEEE report, replace the **old T² distillation results**, not the legitimate GPTQ numbers.

### Final submission checks

- Keep this executed `.ipynb` with its outputs and the new Drive artifact directory.
- Verify the full `teacher_model/` checkpoint is accessible through either the repository or explicit retrieval instructions.
- Verify `runs/<selected run>/best_model/` contains `model.safetensors`, `config.json`, tokenizer files.
- Export/copy essential outputs into your submission repository or provide reproducible links and commands for large model artifacts.
- Create/update the required `RUN_LOG.txt`, `METRICS.md`, `AI_USE.md` and regenerate the IEEE report using the **new real numbers**.
- Check private GitHub repository access and create the `hw12` tag only after the complete evidence is verified.

**Do not change the loss function after running this experiment without repeating affected training.**